In [3]:
"""import os
import json
import json5
from time import sleep
#from openai import OpenAI
from tqdm import tqdm
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from gensim.utils import simple_preprocess"""

'import os\nimport json\nimport json5\nfrom time import sleep\n#from openai import OpenAI\nfrom tqdm import tqdm\nimport numpy as np\nimport pandas as pd\nimport faiss\nfrom sentence_transformers import SentenceTransformer\nfrom gensim.models.doc2vec import Doc2Vec, TaggedDocument\nfrom gensim.utils import simple_preprocess'

## 3. resume extract and conversion from json to text

In [4]:
import os
import json
from tqdm import tqdm
from dotenv import load_dotenv
from langchain_community.llms import Ollama

In [5]:
os.environ.setdefault("LLM_BACKEND", "ollama")
os.environ.setdefault("OLLAMA_MODEL", "mistral")

'mistral'

In [6]:
def weight_resume_llm(cv_text):
    prompt = f"""
    You are an expert resume analyst.

    Your task is to identify the candidate’s CORE PROFESSIONAL PROFILE.
    Focus on clarity and precision. Avoid noise and redundancy.

    RULES (IMPORTANT):
    - Identify ONE primary professional role only
    - Identify at most ONE secondary role (optional)
    - Select NO MORE THAN 6 core skills
    - Select NO MORE THAN 4 experience themes
    - Use short, clear, professional wording
    - Do NOT repeat skills in multiple sections
    - Do NOT add explanations or prose

    OUTPUT FORMAT (STRICT – follow exactly):

    === CORE PROFILE ===
    PRIMARY_ROLE: <short role name>
    SECONDARY_ROLE: <short role name or NONE>

    CORE_SKILLS:
    - <skill> (expert | advanced | intermediate)
    - <skill> (expert | advanced | intermediate)

    CORE_EXPERIENCE_THEMES:
    - <short theme>
    - <short theme>

    KEYWORDS:
    <keyword1> | <keyword2> | <keyword3> | <keyword4> | <keyword5>

    CV:
    ----------------
    {cv_text}
    ----------------

    """
    llm = Ollama(
    model="mistral",
    base_url="http://127.0.0.1:11434",
    timeout=300
    )
    response = llm.invoke(prompt)
    return response


### Conversion without experiences description

In [7]:
def clean_string(s):
    return str(s).replace("\n", " ").strip()

In [8]:
def json_to_text(data):
    parts = []

    skills = data.get("skills")
    if skills:
        skills_list = [clean_string(s) for s in (skills if isinstance(skills, list) else [skills]) if s]
        parts.append("Skills: " + "; ".join(skills_list))

    experience = data.get("experience")
    if experience:
        exp_list = []
        for e in experience:
            title = clean_string(e.get("title", ""))
            company = clean_string(e.get("company", ""))
            years = clean_string(e.get("years", ""))
            exp_list.append(" ".join(filter(None, [title, "at" if title and company else "", company, years])))
        parts.append("Experience: " + "; ".join(exp_list))

    education = data.get("education")
    if education:
        edu_list = []
        for e in education:
            degree = clean_string(e.get("degree", ""))
            school = clean_string(e.get("school", ""))
            edu_list.append(" ".join(filter(None, [degree, "at" if degree and school else "", school])))
        parts.append("Education: " + "; ".join(edu_list))

    certifications = data.get("certifications")
    if certifications:
        cert_list = [clean_string(c) for c in (certifications if isinstance(certifications, list) else [certifications]) if c]
        parts.append("Certifications: " + "; ".join(cert_list))

    summary = data.get("summary")
    if summary:
        parts.append("Summary: " + clean_string(summary))

    base_text="\n".join(parts).strip()
    """    
    try:
        weighted_profile = weight_resume_llm(base_text)
        return base_text + "\n\n" + weighted_profile
    except Exception as e:
        print("⚠️ LLM weighting failed:", e)
        return base_text"""
    return base_text

In [9]:
"""def process_json_folder(input_folder, output_folder, limit=50):
    os.makedirs(output_folder, exist_ok=True)
    json_files = sorted([f for f in os.listdir(input_folder) if f.endswith(".json")])[:limit]
    for filename in tqdm(json_files, desc=f"Processing {len(json_files)} JSON files"):
        input_path = os.path.join(input_folder, filename)
        with open(input_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        output_path = os.path.join(output_folder, f"{os.path.splitext(filename)[0]}.txt")
        with open(output_path, "w", encoding="utf-8") as out:
            out.write(json_to_text(data))"""
def process_json_folder(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    for filename in tqdm([f for f in os.listdir(input_folder) if f.endswith(".json")], desc="Processing JSON files"):
        input_path = os.path.join(input_folder, filename)
        with open(input_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        output_path = os.path.join(output_folder, f"{os.path.splitext(filename)[0]}.txt")
        with open(output_path, "w", encoding="utf-8") as out:
            out.write(json_to_text(data))

### Conversion with experiences description

In [10]:
def json_to_text1(data):
    parts = []
    skills = data.get("skills")
    if skills:
        skills_list = [clean_string(s) for s in (skills if isinstance(skills, list) else [skills]) if s]
        parts.append("Skills: " + "; ".join(skills_list))

    experience = data.get("experience")
    if experience:
        exp_list = []
        for e in experience:
            title = clean_string(e.get("title", ""))
            company = clean_string(e.get("company", ""))
            start_date = clean_string(e.get("start_date", ""))
            end_date = clean_string(e.get("end_date", ""))
            description = clean_string(e.get("description", ""))
            
            exp_parts = []
            if title:
                exp_parts.append(title)
            if company:
                exp_parts.append(f"at {company}")
            if start_date or end_date:
                date_range = f"({start_date} - {end_date})".replace("  ", " ").strip()
                exp_parts.append(date_range)
            if description:
                exp_parts.append(f": {description}")
            
            exp_list.append(" ".join(exp_parts))
        
        parts.append("Experience: " + " | ".join(exp_list))

    education = data.get("education")
    if education:
        edu_list = []
        for e in education:
            degree = clean_string(e.get("degree", ""))
            school = clean_string(e.get("school", ""))
            edu_list.append(" ".join(filter(None, [degree, "at" if degree and school else "", school])))
        parts.append("Education: " + "; ".join(edu_list))

    certifications = data.get("certifications")
    if certifications:
        cert_list = [clean_string(c) for c in (certifications if isinstance(certifications, list) else [certifications]) if c]
        if cert_list:
            parts.append("Certifications: " + "; ".join(cert_list))

    summary = data.get("summary")
    if summary:
        parts.append("Summary: " + clean_string(summary))

    base_text="\n".join(parts).strip()
    """
    try:
        weighted_profile = weight_resume_llm(base_text)
        return base_text + "\n\n" + weighted_profile
    except Exception as e:
        print("⚠️ LLM weighting failed:", e)
        return base_text"""
    return base_text


In [11]:
def process_json_folder1(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    for filename in tqdm([f for f in os.listdir(input_folder) if f.endswith(".json")], desc="Processing JSON files"):
        input_path = os.path.join(input_folder, filename)
        with open(input_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        output_path = os.path.join(output_folder, f"{os.path.splitext(filename)[0]}.txt")
        with open(output_path, "w", encoding="utf-8") as out:
            out.write(json_to_text1(data))

In [12]:
# first approach
process_json_folder("./data/resume_extract_json", "./data/resume_extract_text")

Processing JSON files: 100%|██████████| 2549/2549 [00:28<00:00, 88.53it/s] 


In [13]:
# second approach
process_json_folder1("./data/resume_extract_json", "./data/resume_extract_text1")

Processing JSON files: 100%|██████████| 2549/2549 [00:01<00:00, 1280.23it/s]


## 4. resume embedding

### Embedding with smaller extracted text

In [14]:
import os
import json
import numpy as np
import faiss
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from gensim.utils import simple_preprocess

Embedding with SentenceTransformer

In [15]:
model = SentenceTransformer("all-MiniLM-L6-v2")

input_folder = "./data/resume_extract_text"
txt_files = [f for f in os.listdir(input_folder) if f.endswith(".txt")]

batch_size = 32
embeddings_list = []

for start in tqdm(range(0, len(txt_files), batch_size), desc="Encoding resumes"):
    batch_files = txt_files[start:start + batch_size]
    texts = []
    for file in batch_files:
        path = os.path.join(input_folder, file)
        try:
            with open(path, "r", encoding="utf-8") as f:
                texts.append(f.read())
        except Exception as e:
            print(f"Error reading {file}: {e}")
            texts.append("")

    batch_embeddings = model.encode(texts, convert_to_numpy=True, batch_size=batch_size)
    batch_embeddings /= np.linalg.norm(batch_embeddings, axis=1, keepdims=True)
    embeddings_list.append(batch_embeddings)

all_embeddings = np.vstack(embeddings_list)
faiss_index = faiss.IndexFlatIP(all_embeddings.shape[1])
faiss_index.add(all_embeddings)
faiss.write_index(faiss_index, "./data/resume_index.faiss")

with open("./data/resume_index_mapping.json", "w", encoding="utf-8") as f:
    json.dump(txt_files, f, ensure_ascii=False, indent=4)

Encoding resumes: 100%|██████████| 80/80 [00:30<00:00,  2.66it/s]


Embedding with Doc2Vec

In [16]:
input_folder = "./data/resume_extract_text"
txt_files = [f for f in os.listdir(input_folder) if f.endswith(".txt")]

tagged_data = []
for idx, file in enumerate(tqdm(txt_files, desc="Preparing documents")):
    path = os.path.join(input_folder, file)
    try:
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
            tokens = simple_preprocess(text)
            tagged_data.append(TaggedDocument(words=tokens, tags=[str(idx)]))
    except Exception as e:
        print(f"Error reading {file}: {e}")
        tagged_data.append(TaggedDocument(words=[], tags=[str(idx)]))

print("Training Doc2Vec model...")
model = Doc2Vec(
    vector_size=300,
    window=5,
    min_count=2,
    workers=4, 
    epochs=40,
    dm=1,
    seed=42
)

model.build_vocab(tagged_data)
model.train(tagged_data, total_examples=model.corpus_count, epochs=model.epochs)

print("Generating embeddings...")
all_embeddings = np.array([model.dv[str(i)] for i in range(len(txt_files))], dtype=np.float32)
all_embeddings /= np.linalg.norm(all_embeddings, axis=1, keepdims=True)

faiss_index = faiss.IndexFlatIP(all_embeddings.shape[1])
faiss_index.add(all_embeddings)
faiss.write_index(faiss_index, "./data/resume_index_doc2vec.faiss")

with open("./data/resume_index_mapping_doc2vec.json", "w", encoding="utf-8") as f:
    json.dump(txt_files, f, ensure_ascii=False, indent=4)

model.save("./data/doc2vec_resume.model")

print(f"✅ Doc2Vec embeddings created: {all_embeddings.shape}")

Preparing documents: 100%|██████████| 2549/2549 [00:00<00:00, 3001.85it/s]


Training Doc2Vec model...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Generating embeddings...
✅ Doc2Vec embeddings created: (2549, 300)


### Embedding with bigger extracted text

Embedding with SentenceTransformer

In [17]:
model = SentenceTransformer("all-MiniLM-L6-v2") #"all-mpnet-base-v2"

input_folder = "./data/resume_extract_text1"
txt_files = [f for f in os.listdir(input_folder) if f.endswith(".txt")]

batch_size = 32
embeddings_list = []

for start in tqdm(range(0, len(txt_files), batch_size), desc="Encoding resumes"):
    batch_files = txt_files[start:start + batch_size]
    texts = []
    for file in batch_files:
        path = os.path.join(input_folder, file)
        try:
            with open(path, "r", encoding="utf-8") as f:
                texts.append(f.read())
        except Exception as e:
            print(f"Error reading {file}: {e}")
            texts.append("")

    batch_embeddings = model.encode(texts, convert_to_numpy=True, batch_size=batch_size)
    batch_embeddings /= np.linalg.norm(batch_embeddings, axis=1, keepdims=True)
    embeddings_list.append(batch_embeddings)

all_embeddings = np.vstack(embeddings_list)
faiss_index = faiss.IndexFlatIP(all_embeddings.shape[1])
faiss_index.add(all_embeddings)
faiss.write_index(faiss_index, "./data/resume_index1.faiss")

with open("./data/resume_index_mapping1.json", "w", encoding="utf-8") as f:
    json.dump(txt_files, f, ensure_ascii=False, indent=4)

Encoding resumes: 100%|██████████| 80/80 [00:31<00:00,  2.57it/s]


Embedding with Doc2Vec

In [18]:
input_folder = "./data/resume_extract_text1"
txt_files = [f for f in os.listdir(input_folder) if f.endswith(".txt")]

tagged_data = []
for idx, file in enumerate(tqdm(txt_files, desc="Preparing documents")):
    path = os.path.join(input_folder, file)
    try:
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
            tokens = simple_preprocess(text)
            tagged_data.append(TaggedDocument(words=tokens, tags=[str(idx)]))
    except Exception as e:
        print(f"Error reading {file}: {e}")
        tagged_data.append(TaggedDocument(words=[], tags=[str(idx)]))

print("Training Doc2Vec model...")
model = Doc2Vec(
    vector_size=300,     
    window=5,          
    min_count=2,         
    workers=4,          
    epochs=40,        
    dm=1,  
    seed=42
)

model.build_vocab(tagged_data)
model.train(tagged_data, total_examples=model.corpus_count, epochs=model.epochs)

print("Generating embeddings...")
all_embeddings = np.array([model.dv[str(i)] for i in range(len(txt_files))], dtype=np.float32)
all_embeddings /= np.linalg.norm(all_embeddings, axis=1, keepdims=True)

faiss_index = faiss.IndexFlatIP(all_embeddings.shape[1])
faiss_index.add(all_embeddings)
faiss.write_index(faiss_index, "./data/resume_index_doc2vec1.faiss")

with open("./data/resume_index_mapping_doc2vec1.json", "w", encoding="utf-8") as f:
    json.dump(txt_files, f, ensure_ascii=False, indent=4)

model.save("./data/doc2vec_resume1.model")

print(f"✅ Doc2Vec embeddings created: {all_embeddings.shape}")

Preparing documents: 100%|██████████| 2549/2549 [00:02<00:00, 1245.70it/s]


Training Doc2Vec model...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Generating embeddings...
✅ Doc2Vec embeddings created: (2549, 300)


## 5. job offer embedding

In [19]:
import pandas as pd

In [20]:
path ="./data/job_offer/datasets/ravindrasinghrana/job-description-dataset/versions/1/job_descriptions.csv"
print("Path to dataset files:", path)
df = pd.read_csv(path)
df = df.sample(100000)

fields_to_combine = [
    "Job Id", 
    "Job Title", 
    "Job Description", 
    "skills", 
    "Responsibilities",
    "location"]
df["combined_text"] = df[fields_to_combine].astype(str).agg(" ".join, axis=1)

Path to dataset files: ./data/job_offer/datasets/ravindrasinghrana/job-description-dataset/versions/1/job_descriptions.csv


In [21]:
df.head()

,Job Id,Experience,Qualifications,Salary Range,location,Country,latitude,longitude,Work Type,Company Size,...,Job Title,Role,Job Portal,Job Description,Benefits,skills,Responsibilities,Company,Company Profile,combined_text
787450,1628096998513522,0 to 14 Years,B.Tech,$55K-$92K,Phnom Penh,Cambodia,12.5657,104.9909,Part-Time,27523,...,Landscape Architect,Residential Landscape Designer,USAJOBS,Create aesthetically pleasing and functional l...,"{'Legal Assistance, Bonuses and Incentive Prog...",Landscape design Plant selection and care Hard...,Specialize in designing residential outdoor sp...,CSX,"{""Sector"":""Transportation"",""Industry"":""Railroa...",1628096998513522 Landscape Architect Create ae...
1177962,527104154159677,5 to 11 Years,M.Tech,$62K-$98K,Phnom Penh,Cambodia,12.5657,104.9909,Part-Time,125530,...,Software Architect,Solution Architect,USAJOBS,A Solution Architect designs and develops effe...,"{'Flexible Spending Accounts (FSAs), Relocatio...",Solution design Technical architecture Cloud c...,Design high-level solution architectures for s...,IBM (International Business Machines Corporation),"{""Sector"":""Technology/IT Services"",""Industry"":...",527104154159677 Software Architect A Solution ...
661430,396281146091559,4 to 12 Years,MCA,$56K-$115K,Warsaw,Poland,51.9194,19.1451,Temporary,18575,...,Systems Engineer,Systems Integration Specialist,Idealist,The role of a Systems Integration Specialist i...,"{'Employee Referral Programs, Financial Counse...",Systems integration Integration architecture D...,Integrate various software and hardware system...,Leidos Holdings,"{""Sector"":""IT Services"",""Industry"":""Informatio...",396281146091559 Systems Engineer The role of a...
1277930,310287488169380,1 to 15 Years,B.Tech,$63K-$106K,Bamako,Mali,17.5707,-3.9962,Part-Time,123329,...,Psychologist,Clinical Psychologist,CareerBuilder,Clinical Psychologists diagnose and treat ment...,"{'Employee Assistance Programs (EAP), Tuition ...",Clinical psychology Counseling skills Assessme...,Provide therapy and counseling services to ind...,Seaboard,"{""Sector"":""Transportation"",""Industry"":""Food Pr...",310287488169380 Psychologist Clinical Psycholo...
1387246,1358975344563112,0 to 10 Years,B.Tech,$60K-$85K,Andorra la Vella,Andorra,42.5063,1.5218,Part-Time,68599,...,SEO Specialist,Content SEO Strategist,SimplyHired,Content SEO Strategists create SEO strategies ...,"{'Employee Referral Programs, Financial Counse...",Content strategy SEO content optimization Cont...,"Develop content strategies for SEO, including ...",Alcoa,"{""Sector"":""Metals"",""Industry"":""Metals"",""City"":...",1358975344563112 SEO Specialist Content SEO St...


In [22]:
df.columns

Index(['Job Id', 'Experience', 'Qualifications', 'Salary Range', 'location',
       'Country', 'latitude', 'longitude', 'Work Type', 'Company Size',
       'Job Posting Date', 'Preference', 'Contact Person', 'Contact',
       'Job Title', 'Role', 'Job Portal', 'Job Description', 'Benefits',
       'skills', 'Responsibilities', 'Company', 'Company Profile',
       'combined_text'],
      dtype='object')

### Embedding with SentenceTransformer

In [23]:
model = SentenceTransformer("all-MiniLM-L6-v2")
job_embeddings = model.encode(df["combined_text"].tolist(), convert_to_numpy=True, show_progress_bar=True, normalize_embeddings=True)

faiss_index = faiss.IndexFlatIP(job_embeddings.shape[1])
faiss_index.add(job_embeddings)
faiss.write_index(faiss_index, "./data/jobs_index.faiss")

job_id_mapping = {i: jid for i, jid in enumerate(df["Job Id"].tolist())}
with open("./data/jobs_index_mapping.json", "w", encoding="utf-8") as f:
    json.dump(job_id_mapping, f, indent=4)

Batches:   0%|          | 0/3125 [00:00<?, ?it/s]

### Embedding with Doc2vec

In [24]:
tagged_data = []
job_texts = df["combined_text"].tolist()

for idx, text in enumerate(tqdm(job_texts, desc="Preparing job documents")):
    tokens = simple_preprocess(str(text))
    tagged_data.append(TaggedDocument(words=tokens, tags=[str(idx)]))

print("Training Doc2Vec model for jobs...")
model = Doc2Vec(
    vector_size=300,
    window=5,
    min_count=2,
    workers=4,
    epochs=40,
    dm=1,
    seed=42
)

model.build_vocab(tagged_data)
model.train(tagged_data, total_examples=model.corpus_count, epochs=model.epochs)

print("Generating job embeddings...")
job_embeddings = np.array([model.dv[str(i)] for i in range(len(job_texts))], dtype=np.float32)
job_embeddings /= np.linalg.norm(job_embeddings, axis=1, keepdims=True)
faiss_index = faiss.IndexFlatIP(job_embeddings.shape[1])
faiss_index.add(job_embeddings)
faiss.write_index(faiss_index, "./data/jobs_index_doc2vec.faiss")

job_id_mapping = {i: jid for i, jid in enumerate(df["Job Id"].tolist())}
with open("./data/jobs_index_mapping_doc2vec.json", "w", encoding="utf-8") as f:
    json.dump(job_id_mapping, f, indent=4)

model.save("./data/doc2vec_jobs.model")

print(f"✅ Doc2Vec job embeddings created: {job_embeddings.shape}")

Preparing job documents: 100%|██████████| 100000/100000 [00:08<00:00, 11342.53it/s]


Training Doc2Vec model for jobs...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Generating job embeddings...
✅ Doc2Vec job embeddings created: (100000, 300)


## 6. resume and job offer match

In [25]:
import umap.umap_ as umap
import matplotlib.pyplot as plt

In [26]:
CV_INDEX = [np.random.choice(2549) for _ in range(20)]
TOP_N = 2   

### Matching with smaller embedded resume text

Matching with SentenceTransformer embedding

In [27]:
jobs_index = faiss.read_index("./data/jobs_index.faiss")
with open("./data/jobs_index_mapping.json", "r") as f:
    jobs_mapping = {int(k): v for k, v in json.load(f).items()}

resume_index = faiss.read_index("./data/resume_index.faiss")
with open("./data/resume_index_mapping.json", "r") as f:
    resume_mapping = json.load(f)

model = SentenceTransformer("all-MiniLM-L6-v2")

def reconstruct_embeddings(index):
    embeddings = np.zeros((index.ntotal, index.d), dtype=np.float32)
    for i in range(index.ntotal):
        index.reconstruct(i, embeddings[i])
    return embeddings

resume_json_folder = "./data/resume_extract_text"
resume_data = {}

for fname in os.listdir(resume_json_folder):
    if fname.lower().endswith(".txt"):
        path = os.path.join(resume_json_folder, fname)
        with open(path, "r", encoding="utf-8") as f:
            data = f.read()
        resume_data[fname] = data

jobs_df = pd.read_csv("./data/job_offer/datasets/ravindrasinghrana/job-description-dataset/versions/1/job_descriptions.csv")

cv_embeddings = reconstruct_embeddings(resume_index)
job_embeddings = reconstruct_embeddings(jobs_index)
all_embeddings = np.vstack([cv_embeddings, job_embeddings])
results_ST_small=[]

for cv_index in CV_INDEX:
    cv_filename = resume_mapping[cv_index]

    print("\n===== Selected resume =====")
    if cv_filename in resume_data:
        print(resume_data[cv_filename])

    cv_emb = cv_embeddings[cv_index:cv_index+1]
    distances, indices = jobs_index.search(cv_emb, TOP_N)

    results = [
        {"job_index": int(idx), "job_id": jobs_mapping[int(idx)], "score": float(score)}
        for score, idx in zip(distances[0], indices[0])
    ]

    print(f"\n===== Top {TOP_N} matching job offers =====")
    tmp_score=0
    for r in results:
        job_id = r["job_id"]
        tmp_score+=r['score']
        print(f"\n--- 💼 Found offers : {job_id} ---")
        print(f"Similarity score : {r['score']:.4f}")
        
        job_row = jobs_df[jobs_df["Job Id"] == job_id]

        if len(job_row) == 0:
            print("⚠️ Offer not found in jobs_df")
            continue
        
        row = job_row.iloc[0]
        
        fields_to_show = [
            "Job Title", "Company Name", "Location", "Experience",
            "Qualifications", "Skills", "Job Description", "Responsibilities",
            "Work Type", "Salary Range"
        ]
        
        for col in fields_to_show:
            if col in row and not pd.isna(row[col]):
                print(f"{col}: {row[col]}")
    results_ST_small.append(tmp_score/TOP_N)


===== Selected resume =====
Skills: classroom management; teaching; tutoring; counseling; experience with special needs; parent communication; interactive teaching/learning; innovative lesson planning; team building and leadership; lesson planning; education strategies; plan development; reading assistance; bilingual English/Spanish; teamwork; attention to detail; planning and organizing; multitasking; clear writing; active listening; feedback and collaboration; CPR/First Aid certified
Experience: Lead Teacher at Company Name; Head Teacher at Company Name; case manager at Company Name; Assistant Teacher at Company Name
Education: M.S. in Education/ Special Education Early childhood at Touro College; A.S., Liberal arts at Kingsborough Community College; Bachelor of Science in Psychology at College of Staten Island
Certifications: CPR; First Aid
Summary: Solid background in special needs and early childhood education, with strong emphasis in children's development. Consistently exceed t

Matching with Doc2vec embedding

In [29]:
cv_doc2vec_model = Doc2Vec.load("./data/doc2vec_resume.model")
job_doc2vec_model = Doc2Vec.load("./data/doc2vec_jobs.model")

resume_index = faiss.read_index("./data/resume_index_doc2vec.faiss")
with open("./data/resume_index_mapping_doc2vec.json", "r") as f:
    resume_mapping = json.load(f)

jobs_index = faiss.read_index("./data/jobs_index_doc2vec.faiss")
with open("./data/jobs_index_mapping_doc2vec.json", "r") as f:
    jobs_mapping = {int(k): v for k, v in json.load(f).items()}

def reconstruct_embeddings(index):
    embeddings = np.zeros((index.ntotal, index.d), dtype=np.float32)
    for i in range(index.ntotal):
        index.reconstruct(i, embeddings[i])
    return embeddings

resume_json_folder = "./data/resume_extract_text"
resume_data = {}

for fname in os.listdir(resume_json_folder):
    if fname.lower().endswith(".txt"):
        path = os.path.join(resume_json_folder, fname)
        with open(path, "r", encoding="utf-8") as f:
            data = f.read()
        resume_data[fname] = data

jobs_df = pd.read_csv("./data/job_offer/datasets/ravindrasinghrana/job-description-dataset/versions/1/job_descriptions.csv")

cv_embeddings = reconstruct_embeddings(resume_index)
job_embeddings = reconstruct_embeddings(jobs_index)

all_embeddings = np.vstack([cv_embeddings, job_embeddings])
results_D2V_small=[]

for cv_index in CV_INDEX:
    cv_filename = resume_mapping[cv_index]

    print("\n===== Selected resume =====")
    if cv_filename in resume_data:
        print(resume_data[cv_filename])

    cv_emb = cv_embeddings[cv_index:cv_index+1]
    distances, indices = jobs_index.search(cv_emb, TOP_N)

    results = [
        {"job_index": int(idx), "job_id": jobs_mapping[int(idx)], "score": float(score)}
        for score, idx in zip(distances[0], indices[0])
    ]

    print(f"\n===== Top {TOP_N} matching job offers =====")
    tmp_score=0
    for r in results:
        job_id = r["job_id"]
        tmp_score+=r['score']
        print(f"\n--- 💼 Found offers : {job_id} ---")
        print(f"Similarity score : {r['score']:.4f}")
        
        job_row = jobs_df[jobs_df["Job Id"] == job_id]

        if len(job_row) == 0:
            print("⚠️ Offre not found in jobs_df")
            continue
        
        row = job_row.iloc[0]
        
        fields_to_show = [
            "Job Title", "Company Name", "Location", "Experience",
            "Qualifications", "Skills", "Job Description", "Responsibilities",
            "Work Type", "Salary Range"
        ]
        
        for col in fields_to_show:
            if col in row and not pd.isna(row[col]):
                print(f"{col}: {row[col]}")
    results_D2V_small.append(tmp_score/TOP_N)


===== Selected resume =====
Skills: classroom management; teaching; tutoring; counseling; experience with special needs; parent communication; interactive teaching/learning; innovative lesson planning; team building and leadership; lesson planning; education strategies; plan development; reading assistance; bilingual English/Spanish; teamwork; attention to detail; planning and organizing; multitasking; clear writing; active listening; feedback and collaboration; CPR/First Aid certified
Experience: Lead Teacher at Company Name; Head Teacher at Company Name; case manager at Company Name; Assistant Teacher at Company Name
Education: M.S. in Education/ Special Education Early childhood at Touro College; A.S., Liberal arts at Kingsborough Community College; Bachelor of Science in Psychology at College of Staten Island
Certifications: CPR; First Aid
Summary: Solid background in special needs and early childhood education, with strong emphasis in children's development. Consistently exceed t

### Matching with bigger embedded resume text

Matching with SentenceTransformer embedding

In [30]:
jobs_index = faiss.read_index("./data/jobs_index.faiss")
with open("./data/jobs_index_mapping.json", "r") as f:
    jobs_mapping = {int(k): v for k, v in json.load(f).items()}

resume_index = faiss.read_index("./data/resume_index1.faiss")
with open("./data/resume_index_mapping1.json", "r") as f:
    resume_mapping = json.load(f)

model = SentenceTransformer("all-MiniLM-L6-v2")

def reconstruct_embeddings(index):
    embeddings = np.zeros((index.ntotal, index.d), dtype=np.float32)
    for i in range(index.ntotal):
        index.reconstruct(i, embeddings[i])
    return embeddings

resume_json_folder = "./data/resume_extract_text1"
resume_data = {}

for fname in os.listdir(resume_json_folder):
    if fname.lower().endswith(".txt"):
        path = os.path.join(resume_json_folder, fname)
        with open(path, "r", encoding="utf-8") as f:
            data = f.read()
        resume_data[fname] = data

jobs_df = pd.read_csv("./data/job_offer/datasets/ravindrasinghrana/job-description-dataset/versions/1/job_descriptions.csv")

cv_embeddings = reconstruct_embeddings(resume_index)
job_embeddings = reconstruct_embeddings(jobs_index)
all_embeddings = np.vstack([cv_embeddings, job_embeddings])
results_ST_big=[]

for cv_index in CV_INDEX:
    cv_filename = resume_mapping[cv_index]

    print("\n===== Selected resume =====")
    if cv_filename in resume_data:
        print(resume_data[cv_filename])

    cv_emb = cv_embeddings[cv_index:cv_index+1]
    distances, indices = jobs_index.search(cv_emb, TOP_N)

    results = [
        {"job_index": int(idx), "job_id": jobs_mapping[int(idx)], "score": float(score)}
        for score, idx in zip(distances[0], indices[0])
    ]

    print(f"\n===== Top {TOP_N} matching job offers =====")
    tmp_score=0
    for r in results:
        job_id = r["job_id"]
        tmp_score+=r['score']
        print(f"\n--- 💼 Found offer : {job_id} ---")
        print(f"Similarity score : {r['score']:.4f}")
        
        job_row = jobs_df[jobs_df["Job Id"] == job_id]

        if len(job_row) == 0:
            print("⚠️ Offer not found in jobs_df")
            continue
        
        row = job_row.iloc[0]
        
        fields_to_show = [
            "Job Title", "Company Name", "Location", "Experience",
            "Qualifications", "Skills", "Job Description", "Responsibilities",
            "Work Type", "Salary Range"
        ]
        
        for col in fields_to_show:
            if col in row and not pd.isna(row[col]):
                print(f"{col}: {row[col]}")
    results_ST_big.append(tmp_score/TOP_N)


===== Selected resume =====
Skills: classroom management; teaching; tutoring; counseling; experience with special needs; parent communication; interactive teaching/learning; innovative lesson planning; team building and leadership; lesson planning; education strategies; plan development; reading assistance; bilingual English/Spanish; teamwork; attention to detail; planning and organizing; multitasking; clear writing; active listening; feedback and collaboration; CPR/First Aid certified
Experience: Lead Teacher at Company Name (Aug 2013 - Jun 2015) : Plan and execute daily lessons. Manage children portfolio and progress using Teaching Strategies Gold. Make in-home student referrals. Maintain the comfort, safety and educational demeanor of the classroom environment. Supervise one assistant teacher's in the classroom. Plan and allocate work equally among the staff. Evaluate and test students for appropriate class placement. | Head Teacher at Company Name (Jan 2003 - Aug 2013) : Evaluate 

Matching with Doc2vec embedding

In [31]:
cv_doc2vec_model = Doc2Vec.load("./data/doc2vec_resume1.model")
job_doc2vec_model = Doc2Vec.load("./data/doc2vec_jobs.model")

resume_index = faiss.read_index("./data/resume_index_doc2vec1.faiss")
with open("./data/resume_index_mapping_doc2vec1.json", "r") as f:
    resume_mapping = json.load(f)

jobs_index = faiss.read_index("./data/jobs_index_doc2vec.faiss")
with open("./data/jobs_index_mapping_doc2vec.json", "r") as f:
    jobs_mapping = {int(k): v for k, v in json.load(f).items()}

def reconstruct_embeddings(index):
    embeddings = np.zeros((index.ntotal, index.d), dtype=np.float32)
    for i in range(index.ntotal):
        index.reconstruct(i, embeddings[i])
    return embeddings

resume_json_folder = "./data/resume_extract_text1"
resume_data = {}

for fname in os.listdir(resume_json_folder):
    if fname.lower().endswith(".txt"):
        path = os.path.join(resume_json_folder, fname)
        with open(path, "r", encoding="utf-8") as f:
            data = f.read()
        resume_data[fname] = data

jobs_df = pd.read_csv("./data/job_offer/datasets/ravindrasinghrana/job-description-dataset/versions/1/job_descriptions.csv")

cv_embeddings = reconstruct_embeddings(resume_index)
job_embeddings = reconstruct_embeddings(jobs_index)

all_embeddings = np.vstack([cv_embeddings, job_embeddings])
results_D2V_big=[]

for cv_index in CV_INDEX:
    cv_filename = resume_mapping[cv_index]

    print("===== Selected resume =====")
    if cv_filename in resume_data:
        print(resume_data[cv_filename])

    cv_emb = cv_embeddings[cv_index:cv_index+1]

    distances, indices = jobs_index.search(cv_emb, TOP_N)

    results = [
        {"job_index": int(idx), "job_id": jobs_mapping[int(idx)], "score": float(score)}
        for score, idx in zip(distances[0], indices[0])
    ]

    print(f"\n===== Top {TOP_N} matching job offers =====")
    tmp_score=0
    for r in results:
        job_id = r["job_id"]
        tmp_score+=r['score']
        print(f"\n--- 💼 Found offers : {job_id} ---")
        print(f"Similarity score : {r['score']:.4f}")
        
        job_row = jobs_df[jobs_df["Job Id"] == job_id]

        if len(job_row) == 0:
            print("⚠️ Offre not found in jobs_df")
            continue
        
        row = job_row.iloc[0]
        
        fields_to_show = [
            "Job Title", "Company Name", "Location", "Experience",
            "Qualifications", "Skills", "Job Description", "Responsibilities",
            "Work Type", "Salary Range"
        ]
        
        for col in fields_to_show:
            if col in row and not pd.isna(row[col]):
                print(f"{col}: {row[col]}")
    results_D2V_big.append(tmp_score/TOP_N)

===== Selected resume =====
Skills: classroom management; teaching; tutoring; counseling; experience with special needs; parent communication; interactive teaching/learning; innovative lesson planning; team building and leadership; lesson planning; education strategies; plan development; reading assistance; bilingual English/Spanish; teamwork; attention to detail; planning and organizing; multitasking; clear writing; active listening; feedback and collaboration; CPR/First Aid certified
Experience: Lead Teacher at Company Name (Aug 2013 - Jun 2015) : Plan and execute daily lessons. Manage children portfolio and progress using Teaching Strategies Gold. Make in-home student referrals. Maintain the comfort, safety and educational demeanor of the classroom environment. Supervise one assistant teacher's in the classroom. Plan and allocate work equally among the staff. Evaluate and test students for appropriate class placement. | Head Teacher at Company Name (Jan 2003 - Aug 2013) : Evaluate a

In [32]:
print("Comparison of approaches:")
print("ST with smaller resumes embedding: ", results_ST_small)
print("D2V with smaller resumes embedding: ", results_D2V_small)
print("ST with bigger resumes embedding: ", results_ST_big)
print("D2V with bigger resumes embedding: ", results_D2V_big)

Comparison of approaches:
ST with smaller resumes embedding:  [0.5964137613773346, 0.6209318339824677, 0.5719872713088989, 0.5163479745388031, 0.544945478439331, 0.46320840716362, 0.6294699609279633, 0.4644496589899063, 0.5503432154655457, 0.45751872658729553, 0.49061156809329987, 0.5133502185344696, 0.49668553471565247, 0.461374893784523, 0.4703637510538101, 0.493312269449234, 0.5263482928276062, 0.49442118406295776, 0.5195238590240479, 0.538997232913971]
D2V with smaller resumes embedding:  [0.20927492529153824, 0.23812918365001678, 0.14345034211874008, 0.08224032074213028, 0.17131010442972183, 0.177726611495018, 0.21740764379501343, 0.2201634868979454, 0.17724590748548508, 0.18095484375953674, 0.19698191434144974, 0.19603731483221054, 0.17943106591701508, 0.17934858798980713, 0.20103824138641357, 0.17173238098621368, 0.15561378747224808, 0.187616266310215, 0.2653256356716156, 0.20255490392446518]
ST with bigger resumes embedding:  [0.6123619973659515, 0.6074380278587341, 0.599783331

In [49]:
jobs_index = faiss.read_index("./data/jobs_index.faiss")
with open("./data/jobs_index_mapping.json", "r") as f:
    jobs_mapping = {int(k): v for k, v in json.load(f).items()}

resume_index = faiss.read_index("./data/resume_index.faiss")
with open("./data/resume_index_mapping.json", "r") as f:
    resume_mapping = json.load(f)

model = SentenceTransformer("all-MiniLM-L6-v2")

def reconstruct_embeddings(index):
    embeddings = np.zeros((index.ntotal, index.d), dtype=np.float32)
    for i in range(index.ntotal):
        index.reconstruct(i, embeddings[i])
    return embeddings

resume_json_folder = "./data/resume_extract_text"
resume_data = {}

for fname in os.listdir(resume_json_folder):
    if fname.lower().endswith(".txt"):
        path = os.path.join(resume_json_folder, fname)
        with open(path, "r", encoding="utf-8") as f:
            data = f.read()
        resume_data[fname] = data

jobs_df = pd.read_csv("./data/job_offer/datasets/ravindrasinghrana/job-description-dataset/versions/1/job_descriptions.csv")
result_df_list=[]

cv_embeddings = reconstruct_embeddings(resume_index)
job_embeddings = reconstruct_embeddings(jobs_index)
all_embeddings = np.vstack([cv_embeddings, job_embeddings])
results_ST_small=[]

cv_index=200
cv_filename = resume_mapping[cv_index]

print("\n===== Selected resume =====")
if cv_filename in resume_data:
    print(resume_data[cv_filename])

cv_emb = cv_embeddings[cv_index:cv_index+1]
distances, indices = jobs_index.search(cv_emb, 5)

results = [
    {"job_index": int(idx), "job_id": jobs_mapping[int(idx)], "score": float(score)}
    for score, idx in zip(distances[0], indices[0])
]

print(f"\n===== Top 5 matching job offers =====")
for r in results:
    job_id = r["job_id"]
    print(f"\n--- 💼 Found offers : {job_id} ---")
    print(f"Similarity score : {r['score']:.4f}")
    
    job_row = jobs_df[jobs_df["Job Id"] == job_id]
    job_result=job_row[["Job Title", "Company", "Experience",
        "Qualifications", "skills", "Job Description", "Responsibilities",
        "Work Type", "Salary Range"]]
    result_df_list.append(job_result)

    if len(job_row) == 0:
        print("⚠️ Offer not found in jobs_df")
        continue
    
    row = job_row.iloc[0]
    
    fields_to_show = [
        "Job Title", "Company Name", "Location", "Experience",
        "Qualifications", "Skills", "Job Description", "Responsibilities",
        "Work Type", "Salary Range"
    ]
    
    for col in fields_to_show:
        if col in row and not pd.isna(row[col]):
            print(f"{col}: {row[col]}")

result_df=result_df_list[0]
for i in range(1,len(result_df_list)):
    result_df=pd.concat([result_df,result_df_list[i]],ignore_index=True)


===== Selected resume =====
Skills: Leadership; Communication; Business operations; Client account management; Budgeting; Negotiation; Employee relations; Self-motivation; Market research and analysis; Customer orientation; Microsoft Family Products; Customer CRM; GPO and IDN targeting; Vendor and Distributor Relations; National Business Development; Regional Business Development; Local Business Development; Forecasting; C-Suite Executive Targeting; Sales Management; Problem Solving
Experience: Director of National Sales- US. Healthcare at Company Name; National Accounts Manager- Northeast Region at Company Name; Business Development Manager at Company Name
Education: B.A. in Marketing at Bloomsburg University
Certifications: Karrass Effective Negotiating Seminar; Linde Pro Sales Training; Sales Performance International-Solution Sales; Sales Performance Internal-Management Training; Challenger Sales Training; Completed Advanced Sales Training I; Consultative Sales Training; Situation

In [50]:
results

[{'job_index': 50229, 'job_id': 1750518493773044, 'score': 0.6867740154266357},
 {'job_index': 47044, 'job_id': 991876011802800, 'score': 0.6814556121826172},
 {'job_index': 30386, 'job_id': 336831904414436, 'score': 0.6803520917892456},
 {'job_index': 28912, 'job_id': 1860884116431098, 'score': 0.6798303723335266},
 {'job_index': 85050, 'job_id': 1759480494318066, 'score': 0.6791889667510986}]

In [51]:
result_df

,Job Title,Company,Experience,Qualifications,skills,Job Description,Responsibilities,Work Type,Salary Range
0,Business Development Manager,ITC Limited,5 to 10 Years,PhD,Sales management Sales strategy development Te...,A Sales Manager leads and guides the sales tea...,"Lead and manage a sales team, set sales goals,...",Contract,$65K-$92K
1,Business Development Manager,Dell Technologies,4 to 8 Years,BA,Sales management Sales strategy development Te...,A Sales Manager leads and guides the sales tea...,"Lead and manage a sales team, set sales goals,...",Contract,$58K-$82K
2,Business Development Manager,Ford Motor Company,3 to 12 Years,BCA,Sales management Sales strategy development Te...,A Sales Manager leads and guides the sales tea...,"Lead and manage a sales team, set sales goals,...",Contract,$55K-$96K
3,Business Development Manager,Principal Financial,2 to 11 Years,BBA,Sales management Sales strategy development Te...,A Sales Manager leads and guides the sales tea...,"Lead and manage a sales team, set sales goals,...",Full-Time,$65K-$112K
4,Business Development Manager,United States Steel,0 to 13 Years,M.Com,Sales management Sales strategy development Te...,A Sales Manager leads and guides the sales tea...,"Lead and manage a sales team, set sales goals,...",Intern,$64K-$125K


In [57]:
prompt = f"""
You are an experienced recruiter.

A candidate has the following resume:

{resume_data[cv_filename]}

Based on vector similarity search, the following job offers were selected and are in this dataframe: {result_df}

Your task is ONLY to explain and interpret these results.

For EACH job offer selected, meaning each line of the dataset:
1. Explain why the resume is relevant
2. Highlight matching skills and experience
3. Mention potential gaps or risks
4. Give a short, honest recommendation

DO NOT re-score.
DO NOT re-rank.
DO NOT invent new jobs.
"""
ollama_model = os.environ.get("OLLAMA_MODEL", "mistral")
llm = Ollama(model=ollama_model)

explanation = llm.invoke(prompt)
print(explanation)

 1. **Job Title: Business Development Manager at ITC Limited**
   - Relevance: The candidate's skills and experience align well with the role of a Business Development Manager, particularly in sales management, client account management, negotiation, employee relations, business operations, and budgeting.
   - Matching Skills and Experience: Leadership, Communication, Sales Management, Business Operations, Client Account Management, Budgeting, Negotiation, Employee Relations. The candidate's experience as a Director of National Sales and Business Development Manager at various companies demonstrates relevant experience.
   - Potential Gaps or Risks: It appears that the candidate does not have specific experience in the pharmaceutical or FMCG industry where ITC Limited operates, which could be a potential risk to the company. However, given the transferable nature of sales and business development skills across industries, this gap might not significantly impact their ability to perform